# Contest inference profile (EVA02 extract cycle)

Organizer latency/FPS is **not** a raw `model()` microbench and **not** retrieval.

Timed: batch-1 `extract()` (organizer one-vehicle cycle) and DataLoader throughput (`extract_frame`, same as Docker). Shared decode lives in `dataset.images`. Untimed: gallery search, re-ranking, AQE/DBA (those scale with gallery size).

Serving kernels (applied in `prepare_inference_model`): fused SDPA after RoPE (`model.attn_kernel=sdpa`), `torch.compile(mode=reduce-overhead)` of an embedding wrapper. Decode/crop/preprocess is the same `VehicleDataset` / DataLoader path as Docker `eval.py`. Compile is paid on first forward, not in timed extract.

| Metric | Protocol |
| --- | --- |
| `latency_b1` | median of 300 full cycles at batch=1 after 50 warmups; CUDA sync before and after every timed sample |
| `throughput` | sustained images/s at batches 1 / 8 / 16 / 32, each window ≥ 10 s; score is the best FPS |
| Weights | sum of `.pt .pth .bin .onnx .engine .plan .safetensors .ckpt .trt .pb .tflite .npz` under the solution dirs; cap **2 GiB** |
| Also logged | peak VRAM, weight-load time (reference), two-run determinism |
| Performance | `10% × latency_score + 10% × throughput_score`. ≤40 ms and ≥100 FPS are full marks; >80 ms or <50 FPS are zero. Files over 2 GiB are not admitted. |

A Lightning training `.ckpt` (optimizer, loops, raw+EMA) is not the contest payload. Export EMA with `scripts/export_serving.py` to `weights/finetuned/eva02.pt`. CatBoost `.cbm` is listed here but is **outside** the official suffix glob.

Set `PROFILE_FAST=1` to shorten warmup/repeats/windows. Default is the contest protocol.


In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hydra import compose, initialize_config_dir

from augmentations import build_transforms
from dataset.images import decode_backend_name, image_path
from eval import load_model
from models.kernels import prepare_inference_model
from profiling import (
    CONTEST_BATCH_SIZES,
    CONTEST_MIN_SECONDS,
    CONTEST_REPEATS,
    CONTEST_WARMUP,
    LIMIT_BYTES,
    STAGES,
    as_jsonable,
    bytes_text,
    inventory,
    performance_scores,
    pick_device,
    profile_extract,
    time_load,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "eva02_k4"
FOLD = 0
VAL = CV / f"fold{FOLD}" / "val"
SERVE = ROOT / "weights" / "finetuned" / "eva02.pt"
FIG_DIR = ROOT / "notebooks" / "eva02" / "inference_profile_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
FAST = os.environ.get("PROFILE_FAST") == "1"
WARMUP = 5 if FAST else CONTEST_WARMUP
REPEATS = 20 if FAST else CONTEST_REPEATS
MIN_SECONDS = 1.0 if FAST else CONTEST_MIN_SECONDS
BATCH_SIZES = (1, 8) if FAST else CONTEST_BATCH_SIZES
STAGE_REPEATS = 4 if FAST else 16
DEVICE = pick_device(prefer=(2,), exclude=(0, 1))
print("ROOT", ROOT)
print("device", DEVICE)
print(
    "protocol",
    {
        "warmup": WARMUP,
        "repeats": REPEATS,
        "min_seconds": MIN_SECONDS,
        "batch_sizes": BATCH_SIZES,
        "fast": FAST,
    },
)

/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Please 'pip install xformers'
Please 'pip install xformers'


ROOT /data/projects/ryzhichkin/ReID
device cuda:2
protocol {'warmup': 50, 'repeats': 300, 'min_seconds': 10.0, 'batch_sizes': (1, 8, 16, 32), 'fast': False}


## Weight inventory vs 2 GiB

Scan `weights/finetuned/` only. That is the contest payload (`eva02.pt`). The training Lightning `.ckpt` is logged as a comparison and is **not** submitted.


In [3]:
metrics_path = VAL / "metrics.json"
ckpt_path = Path(json.loads(metrics_path.read_text())["checkpoint"]) if metrics_path.is_file() else None
roots = [ROOT / "weights" / "finetuned"]
weights = inventory(roots)
files = pd.DataFrame(weights["files"] + weights["extra_files"])
if len(files):
    files["mib"] = files["bytes"] / 1024**2
    files["official"] = files["suffix"].isin(set(item["suffix"] for item in weights["files"]))
    display(
        files.sort_values("bytes", ascending=False)[["name", "suffix", "bytes", "mib", "official", "path"]]
    )
else:
    print("no weight files yet under", roots)
print(
    "official",
    bytes_text(weights["total_bytes"]),
    "limit",
    bytes_text(weights["limit_bytes"]),
    "headroom",
    bytes_text(weights["headroom_bytes"]),
    "within_2gib",
    weights["within_limit"],
    "extra_unscanned",
    bytes_text(weights["extra_bytes"]),
)
if ckpt_path is not None and ckpt_path.is_file():
    print("training lightning ckpt (not submitted)", ckpt_path, bytes_text(ckpt_path.stat().st_size))

                 name suffix       bytes       mib  official                                                                 path
0            eva02.pt    .pt  1228959035 1172.0267      True            /data/projects/ryzhichkin/ReID/weights/finetuned/eva02.pt
1  eva02_catboost.cbm   .cbm       87616    0.0836     False  /data/projects/ryzhichkin/ReID/weights/finetuned/eva02_catboost.cbm
official 1.14 GiB limit 2.00 GiB headroom 875.97 MiB within_2gib True extra_unscanned 85.56 KiB
training lightning ckpt (not submitted) /data/projects/ryzhichkin/ReID/runs/cv/eva02_k4/fold0/checkpoints/epoch024.ckpt 4.55 GiB


## Load serving model

Hydra `experiment=current_best_tuned`, `weights/finetuned/eva02.pt` (exported EMA), `eval.precision=bf16`, TTA off. Device picker skips GPUs 0–1 and prefers `cuda:2` when it is ≥95% free. After `load_model`, `prepare_inference_model` enables fused SDPA, `torch.compile(reduce-overhead)`, and `eval.fast_kernels`.


In [4]:
REPORT = None
MODEL = None
CFG = None
if not SERVE.is_file():
    print("skip load: weights/finetuned/eva02.pt missing")
elif DEVICE.type != "cuda":
    print("skip load: no free CUDA device (refusing CPU EVA02-L)")
else:
    with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs")):
        hydra_cfg = compose(
            config_name="config",
            overrides=[
                f"checkpoint={SERVE}",
                "experiment=current_best_tuned",
                f"eval.device={DEVICE}",
                "eval.precision=bf16",
                "eval.weights=auto",
            ],
        )

    def _load():
        model, cfg, checkpoint, choice = load_model(hydra_cfg, override_items=[])
        model.to(DEVICE).eval()
        model = prepare_inference_model(model, cfg)
        return model, cfg, checkpoint, choice

    packed, load_ms, load_snap = time_load(_load, DEVICE)
    MODEL, CFG, CHECKPOINT, CHOICE = packed
    print(
        "weights",
        CHOICE,
        "load",
        f"{load_ms:.0f} ms",
        "alloc",
        bytes_text(load_snap["allocated_bytes"]),
        "peak",
        bytes_text(load_snap["peak_bytes"]),
    )
    print(
        "pooling",
        CFG.model.pooling.kind,
        "size",
        list(CFG.data.image_size),
        "context",
        float(CFG.data.context_pct),
        "precision",
        CFG.eval.precision,
        "tta",
        bool(CFG.eval.tta.enabled),
    )

weights ema load 4735 ms alloc 1.14 GiB peak 1.14 GiB
pooling attn size [336, 336] context 6.201741080661072 precision bf16 tta False


## Run extract profile

Sample the fold-0 val query list (file order). Gallery search is not in this loop.


In [5]:
if MODEL is None or CFG is None:
    print("skip profile")
else:
    query = pd.read_csv(VAL / "query.csv", dtype={"image_id": str})
    n = max(BATCH_SIZES)
    sample = query.head(n).copy()
    if len(sample) < n:
        raise ValueError(f"need {n} query rows, got {len(sample)}")
    paths = [image_path(ROOT / "data" / "images", image_id) for image_id in sample.image_id]
    bboxes = sample[["x", "y", "w", "h"]].to_numpy().tolist()
    transform = build_transforms(CFG, train=False)
    REPORT = profile_extract(
        roots=roots,
        paths=paths,
        bboxes=bboxes,
        transform=transform,
        model=MODEL,
        device=DEVICE,
        load_ms=load_ms,
        checkpoint=CHECKPOINT,
        weights=CHOICE,
        context_pct=float(CFG.data.context_pct),
        precision=str(CFG.eval.precision),
        tta=CFG.eval.tta,
        model_cfg=CFG.model,
        decode_backend=decode_backend_name(CFG),
        cfg=CFG,
        warmup=WARMUP,
        repeats=REPEATS,
        batch_sizes=BATCH_SIZES,
        min_seconds=MIN_SECONDS,
        vram_repeats=2,
        stage_repeats=STAGE_REPEATS,
    )
    (FIG_DIR / "report.json").write_text(json.dumps(as_jsonable(REPORT), indent=2))
    c = REPORT["contest"]
    board = pd.DataFrame(
        [
            {"metric": "latency_b1_ms", "value": c["latency_b1_ms"], "note": "median full cycle, batch=1"},
            {
                "metric": "throughput_best_fps",
                "value": c["throughput_best_fps"],
                "note": f"best at batch {c['throughput_best_batch_size']}",
            },
            {"metric": "peak_vram", "value": c["peak_vram_bytes"], "note": bytes_text(c["peak_vram_bytes"])},
            {"metric": "load_ms", "value": c["load_ms"], "note": "reference, not scored"},
            {
                "metric": "official_weight_bytes",
                "value": c["weight_bytes"],
                "note": bytes_text(c["weight_bytes"]),
            },
            {
                "metric": "serving_tensor_bytes",
                "value": c["serving_tensor_bytes"],
                "note": bytes_text(c["serving_tensor_bytes"]),
            },
            {"metric": "within_2gib", "value": float(c["within_2gib"]), "note": bytes_text(LIMIT_BYTES)},
            {
                "metric": "performance_eligible",
                "value": float(c["performance_eligible"]),
                "note": "official weight files vs 2 GiB",
            },
            {"metric": "latency_score", "value": c["latency_score"], "note": "1 if <=40 ms, 0 if >80 ms"},
            {
                "metric": "throughput_score",
                "value": c["throughput_score"],
                "note": "1 if >=100 FPS, 0 if <50",
            },
            {
                "metric": "performance_score",
                "value": c["performance_score"],
                "note": "0.10*lat + 0.10*fps (max 0.20)",
            },
            {"metric": "deterministic", "value": float(c["deterministic"]), "note": REPORT["determinism"]},
        ]
    )
    display(board)
    ema_ok = c["serving_tensor_bytes"] <= LIMIT_BYTES
    print(
        "EMA-only performance",
        performance_scores(c["latency_b1_ms"], c["throughput_best_fps"], within_limit=ema_ok),
    )
    display(pd.DataFrame([REPORT["stages_per_image_ms"]]))
    display(pd.DataFrame(REPORT["throughput"]["by_batch"]))
    display(pd.DataFrame(REPORT["vram"]))
    print("determinism", REPORT["determinism"])
    if REPORT["checkpoint"] is not None:
        keys = REPORT["checkpoint"]["bytes_by_key"]
        display(
            pd.DataFrame(
                {
                    "key": list(keys),
                    "bytes": list(keys.values()),
                    "text": [bytes_text(v) for v in keys.values()],
                }
            )
            .sort_values("bytes", ascending=False)
            .head(12)
        )

                   metric           value                                                                                                                    note
0           latency_b1_ms         15.5160                                                                                              median full cycle, batch=1
1     throughput_best_fps        722.1917                                                                                                        best at batch 16
2               peak_vram 1266721280.0000                                                                                                                1.18 GiB
3                 load_ms       4734.6523                                                                                                   reference, not scored
4   official_weight_bytes 1228959035.0000                                                                                                                1.14 GiB
5    serving_tensor_bytes 12

## Stage costs (batch=1, CUDA-synced)

Each bar is measured after the contest warmup. CUDA still syncs before and after that stage, so the sum is **larger** than contest `latency_b1` (one sync around the whole cycle). Use this chart for *where* time goes after warmup, not as the scored number.


In [6]:
if REPORT is None:
    print("skip figure")
else:
    stages = REPORT["stages_per_image_ms"]
    fig, ax = plt.subplots()
    names = list(STAGES)
    vals = [stages[name] for name in names]
    ax.bar(names, vals, color="#3b6ea5")
    ax.set_ylabel("ms / image")
    ax.set_title("extract() stage breakdown (mean of timed batch=1)")
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    fig.savefig(FIG_DIR / "stages.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print("wrote", FIG_DIR / "stages.png")

wrote /data/projects/ryzhichkin/ReID/notebooks/eva02/inference_profile_figs/stages.png


![stages](inference_profile_figs/stages.png)


## latency_b1 distribution


In [7]:
if REPORT is None:
    print("skip figure")
else:
    samples = np.asarray(REPORT["latency"]["samples_ms"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(samples, bins=min(40, max(8, len(samples) // 5)), color="#3b6ea5", edgecolor="white")
    axes[0].axvline(
        REPORT["latency"]["p50_ms"], color="#c0392b", label=f"p50 {REPORT['latency']['p50_ms']:.2f} ms"
    )
    axes[0].axvline(
        REPORT["latency"]["p90_ms"], color="#d68910", label=f"p90 {REPORT['latency']['p90_ms']:.2f} ms"
    )
    axes[0].set_xlabel("ms")
    axes[0].set_ylabel("runs")
    axes[0].set_title("batch=1 full-cycle latency")
    axes[0].legend()
    axes[1].boxplot(samples, vert=True, widths=0.5)
    axes[1].set_ylabel("ms")
    axes[1].set_title("latency_b1 spread")
    fig.savefig(FIG_DIR / "latency.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print(
        "p50",
        REPORT["latency"]["p50_ms"],
        "p90",
        REPORT["latency"]["p90_ms"],
        "p99",
        REPORT["latency"]["p99_ms"],
        "mean",
        REPORT["latency"]["mean_ms"],
    )

p50 15.51597286015749 p90 15.944953076541424 p99 16.069078277796507 mean 15.539261816690365


/tmp/ipykernel_2237332/2088493395.py:17: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  axes[1].boxplot(samples, vert=True, widths=0.5)


![latency](inference_profile_figs/latency.png)


## Throughput vs batch size


In [8]:
if REPORT is None:
    print("skip figure")
else:
    rows = pd.DataFrame(REPORT["throughput"]["by_batch"])
    fig, ax = plt.subplots()
    ax.bar([str(v) for v in rows.batch_size], rows.fps, color="#1e8449")
    ax.set_xlabel("batch size")
    ax.set_ylabel("images / s")
    ax.set_title("sustained extract() FPS (best bar is the contest throughput score)")
    for i, fps in enumerate(rows.fps):
        ax.text(i, fps, f"{fps:.1f}", ha="center", va="bottom", fontsize=8)
    fig.savefig(FIG_DIR / "throughput.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(rows)

   batch_size  images  seconds      fps  ms_per_image
0           1    1407  10.0011 140.6851        7.1081
1           8    6464  10.0071 645.9439        1.5481
2          16    7232  10.0140 722.1917        1.3847
3          32    7200  10.0208 718.5087        1.3918


![throughput](inference_profile_figs/throughput.png)


## Peak VRAM vs batch size


In [9]:
if REPORT is None:
    print("skip figure")
else:
    rows = pd.DataFrame(REPORT["vram"])
    rows["peak_mib"] = rows.peak_bytes / 1024**2
    rows["alloc_mib"] = rows.allocated_bytes / 1024**2
    fig, ax = plt.subplots()
    x = np.arange(len(rows))
    ax.bar(x - 0.18, rows.peak_mib, 0.36, label="peak", color="#6c3483")
    ax.bar(x + 0.18, rows.alloc_mib, 0.36, label="allocated", color="#a569bd")
    ax.set_xticks(x, [str(v) for v in rows.batch_size])
    ax.set_xlabel("batch size")
    ax.set_ylabel("MiB")
    ax.set_title("CUDA memory after extract() (weights already loaded)")
    ax.legend()
    fig.savefig(FIG_DIR / "vram.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(rows[["batch_size", "peak_bytes", "allocated_bytes", "reserved_bytes", "peak_mib"]])

   batch_size  peak_bytes  allocated_bytes  reserved_bytes  peak_mib
0           1  1223931904       1222525440      2344615936 1167.2324
1           8  1233402368       1222525440      2344615936 1176.2642
2          16  1244279296       1222525440      2344615936 1186.6372
3          32  1266721280       1222525440      2344615936 1208.0396


![vram](inference_profile_figs/vram.png)


## Weight bytes vs 2 GiB cap


In [10]:
fig, ax = plt.subplots()
official = weights["total_bytes"]
serving = 0 if REPORT is None else REPORT["contest"]["serving_tensor_bytes"]
labels = ["official files", "EMA/raw tensors", "2 GiB cap"]
vals = [official / 1024**3, serving / 1024**3, LIMIT_BYTES / 1024**3]
colors = ["#c0392b" if official > LIMIT_BYTES else "#1e8449", "#3b6ea5", "#7f8c8d"]
ax.barh(labels, vals, color=colors)
ax.set_xlabel("GiB")
ax.set_title("submit weight budget")
for i, v in enumerate(vals):
    ax.text(v, i, f" {v:.2f} GiB", va="center")
fig.savefig(FIG_DIR / "weights.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(
    "official", bytes_text(official), "serving_tensors", bytes_text(serving), "cap", bytes_text(LIMIT_BYTES)
)

official 1.14 GiB serving_tensors 1.14 GiB cap 2.00 GiB


![weights](inference_profile_figs/weights.png)


## Takeaways

Serving `weights/finetuned/eva02.pt` (labeled-only K=4 full-retrain EMA), bf16, TTA off, isolated `cuda:2` on NVIDIA H200 (torch 2.8.0+cu128, CUDA 12.8, driver 575.57.08). Contest protocol (50 warmup + 300 timed, ≥10 s/batch). Kernels: fused SDPA after RoPE, `torch.compile(mode=reduce-overhead)` of an embedding wrapper. Decode is `VehicleDataset` / DataLoader, same as Docker.

| Metric | Value |
| --- | --- |
| `latency_b1` | **15.5 ms** (p90 15.9 ms, p99 16.1 ms), sequential `extract()` |
| `throughput` | **722 FPS** at batch 16 on the DataLoader `extract_frame` path (batch 32 is 719 FPS) |
| Peak VRAM | 1.14 GiB @ b1 → **1.18 GiB @ b32** (weights already loaded) |
| Load | 4.7 s wrap; compile graph is paid on first forward and excluded from timed extract |
| Serving `eva02.pt` | **~1.14 GiB** (EMA tensors + cfg; under the 2 GiB cap) |
| Training Lightning `.ckpt` | 4.55 GiB (optimizer ~2.27 GiB; **not submitted**) |
| Determinism | bit-identical on two compiled extracts |
| Official performance | latency_score **1.0**, throughput_score **1.0**, performance_score **0.20** |

- Scoring uses **full extract()** for latency, not isolated forward. Decode (~7.7 ms) dominates batch-1; compiled forward is ~5.7 ms. Gallery search and rerank are untimed.
- Submit `weights/finetuned/eva02.pt` from `scripts/export_serving.py`. Do not copy the training Lightning dump into the image or the run is excluded from the 20% performance score.
- Math vs SDPA embeddings were bit-identical. Forced Flash/efficient SDPA did not beat H200 math at batch=1 (~27 ms cycle); `torch.compile` did (15.5 ms). Compile vs eager min cosine 0.99988 on 32 crops — keep compile for serve. Most compile cost is first forward, not `torch.compile(...)`.
- Docker and throughput now share `extract_frame`. The old 8-thread `extract()` pool is gone. Next: persistent workers, overlap host prep with GPU, fewer PIL→NumPy→Tensor copies, then CUDA JPEG (`decode_jpeg(..., device="cuda")`), static compile shapes, TensorRT.
- Container is CUDA 12.8; organizers list a 12.2 driver. NVIDIA documents minor-compatibility limits — confirm by running.
- Stage bars run after the contest warmup. Extra per-stage CUDA syncs still keep the chart diagnostic. `latency_b1` is the harness number.
